### Concatenate the metadata files to a final version

In [1]:
import ast
from importlib import reload
import os
import pandas as pd
import sys

sys.path.append('../helpful_functions/')
import helpful_functions as hf
reload(hf)

<module 'helpful_functions' from '/home/kisa/coding/80K_MPRA/80K-Analysis/07_quality_control/notebooks/control_metadata/../helpful_functions/helpful_functions.py'>

In [2]:
# remove adapters from the design file
fasta_with_adapter = "/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/resources/association_data/design_removed_spaces_deduplicated_sequences_renamed_replaced_comma_with_tilde_greater_to_star_no_brackets.fa"
fasta_with_adapter_df = hf.fasta_to_dataframe(fasta_with_adapter)

In [3]:
fasta_with_adapter_df["sequence"].str.len().value_counts()

sequence
300    80210
298        5
Name: count, dtype: int64

In [4]:
fasta_with_adapter_df["seq_no_adapter"] = fasta_with_adapter_df["sequence"].apply(lambda seq: seq[15:-15])
fasta_with_adapter_df["seq_no_adapter"].str.len().value_counts()

seq_no_adapter
270    80210
268        5
Name: count, dtype: int64

In [5]:
fasta_with_adapter_df['label'] = fasta_with_adapter_df['header'].apply(hf.get_label)

In [6]:
fasta_with_adapter_df

,header,sequence,seq_no_adapter,label
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTAAGAATACAAGTAACTGATGAATGAAGGGGG...,AAGAATACAAGTAACTGATGAATGAAGGGGGCATCTTGTGTCCCCA...,cardiac_neuro_cava_random
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTTTGGGTATGCTGCCCCCCAGCTGGCGGGGCA...,TTGGGTATGCTGCCCCCCAGCTGGCGGGGCACCGGGGACAGGCACA...,cardiac_neuro_cava_random
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTACGAGCAAGGGAATGAGAGAGAGTGGGTTAG...,ACGAGCAAGGGAATGAGAGAGAGTGGGTTAGAGAGTGAGTGAGCCA...,cardiac_neuro_cava_random
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCGTGGACACGCGTGATTGACCCTTTAACTGT...,CGTGGACACGCGTGATTGACCCTTTAACTGTATCCTTAACCACCGC...,cardiac_neuro_cava_random
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCCGGAGAGTCTCAGCTCCCGCAGCCCTAACA...,CCGGAGAGTCTCAGCTCCCGCAGCCCTAACAAACGACCACAGACCT...,cardiac_neuro_cava_random
...,...,...,...,...
80210,MK:tile_2240|chr1-116244322+116244591|scramble...,AGGACCGGATCAACTCTTAATCAAATAACCCATTAATTCTATATAT...,CTTAATCAAATAACCCATTAATTCTATATATCTACCTAATATTAAT...,MK
80211,MK:tile_6675|chr11-2374617+2374886|scramble_ne...,AGGACCGGATCAACTCATCGGCCCTGGTGAAGCGTCCGTCCAGACG...,CATCGGCCCTGGTGAAGCGTCCGTCCAGACGGGCCTGCCTAGCCTC...,MK
80212,MK:tile_18415|chr17-71181691+71181960|scramble...,AGGACCGGATCAACTTAAATATTCAGCGATACATTCCTATTCTTTT...,TAAATATTCAGCGATACATTCCTATTCTTTTTCAGAAGTAGTTATT...,MK
80213,MK:tile_14356|chr15-67031618+67031887|scramble...,AGGACCGGATCAACTTGAAGCCCCTGATTCTGTTAGAATAAGGTTA...,TGAAGCCCCTGATTCTGTTAGAATAAGGTTACTGAGTCGTGTATAC...,MK


In [ ]:
# remove adapter for MPRAsnakeflow run
# no_adapter_output = "/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/resources/association_data/design_removed_spaces_deduplicated_sequences_renamed_replaced_comma_with_tilde_greater_to_star_no_brackets_no_adapter.fa"
# hf.write_fasta(fasta_with_adapter_df, no_adapter_output, header=['header', 'seq_no_adapter'])

# no_adapter_output_label = "/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/resources/association_data/design_removed_spaces_deduplicated_sequences_renamed_replaced_comma_with_tilde_greater_to_star_no_brackets_no_adapter_label.tsv"
# fasta_with_adapter_df[['header', 'label']].to_csv(no_adapter_output_label, sep="\t", index=False, header=None)

: 

: 

In [7]:
# Function to safely evaluate string representations of lists
def safe_eval(x):
    if pd.isna(x):
        return None
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        return x

# column names
col_name = 'name'
col_header = 'name'
col_sequence = 'sequence'
col_category = 'category'
col_class = 'class'
col_source = 'source'
col_ref = 'ref'
col_chr = 'chr'
col_start = 'start'
col_end = 'end'
col_strand = 'strand'
col_variant_class = 'variant_class'
col_variant_pos = 'variant_pos'
col_SPDI = 'SPDI'
col_allele = 'allele'
col_info = 'info'


list_columns = [col_variant_class, col_variant_pos, col_SPDI, col_allele]

In [8]:
metadata_path = "/home/kisa/coding/80K_MPRA/80K-Analysis/07_quality_control/notebooks/control_metadata/MPRA_80215.metadata.tsv.gz"
metadata_df = pd.read_csv(metadata_path, sep='\t', compression='gzip')

# use safe eval on list
# Apply the safe_eval function to the specified columns
for col in list_columns:
    metadata_df[col] = metadata_df[col].apply(safe_eval)

In [ ]:
# write col_name and sequence
output_path = "/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/resources/association_data/metadata_design_no_adapter.fa"

# hf.write_fasta(metadata_df, output_path, header=['header', 'sequence'])

TypeError: write_fasta() missing 1 required positional argument: 'output_path'

In [9]:
# get the labels
design_file = "/data/cephfs-1/home/users/kisa11_c/unmirrored/projects/MPRA/IGVF_Y1_design/design/final_design/results/final_design/design.fa.gz"
design_file = "/home/kisa/coding/80K_MPRA/design_data/design_info/design.fa.gz"
design_df = hf.fasta_to_dataframe(design_file)
design_df['label'] = design_df['header'].apply(hf.get_label)
print("Number unique sequences: ", design_df['sequence'].nunique()) # 80215
print("Number of unique header: ", design_df['header'].nunique()) # 80084
print("Number of labels: ", design_df['label'].nunique()) # 29

Number unique sequences:  80215
Number of unique header:  80084
Number of labels:  29


In [10]:
# get the list of the metadata files
directory_path = '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format'
metadata_file_ending = '.metadata.tsv.gz'

def list_metadata_files_in_subdirectories(directory, metadata_file_ending):
    """Returns all the files in the subdirectories of a given directory"""
    file_list = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            if metadata_file_ending in file:
                file_list.append(os.path.join(root, file))
                # print(os.path.join(root, file))
    return file_list

metadata_file_list_raw = list_metadata_files_in_subdirectories(directory_path, metadata_file_ending)

In [11]:
metadata_file_list_raw

['/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/MPRA_80215.metadata.tsv.gz',
 '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/GC_Mohlke/GC_Mohlke.metadata.tsv.gz',
 '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/C_positive_heart_CAD/C_positive_heart_CAD.metadata.tsv.gz',
 '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/C_positive_heart_AB/C_positive_heart_AB.metadata.tsv.gz',
 '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/GC_Glut_Chengyu/GC_Glut_Chengyu.metadata.tsv.gz',
 '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/C_negative_heart_MK/C_negative_heart_MK.metadata.tsv.gz',
 '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/GC_Liang/GC_Liang.metadata.tsv.gz',
 '/data/cephfs-2/unmirrored/groups

In [13]:
metadata_file_list = ['/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/GC_Mohlke/GC_Mohlke.metadata.tsv.gz',
 '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/C_positive_heart_CAD/C_positive_heart_CAD.metadata.tsv.gz',
 '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/C_positive_heart_AB/C_positive_heart_AB.metadata.tsv.gz',
 '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/GC_Glut_Chengyu/GC_Glut_Chengyu.metadata.tsv.gz',
 '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/C_negative_heart_MK/C_negative_heart_MK.metadata.tsv.gz',
 '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/GC_Liang/GC_Liang.metadata.tsv.gz',
 '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/GC_GABA_Chengyu/GC_GABA_Chengyu.metadata.tsv.gz',
 '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/SLEA/SLEA.metadata.tsv.gz',
 '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/DNase_NegControls/GC_DNase_negative_brain.metadata.tsv.gz',
 '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/DNase_NegControls/GC_DNase_negative_brain_shuffeled.metadata.tsv.gz',
 '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/DNase_NegControls/GC_DNase_negative_blood.metadata.tsv.gz',
 '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/DNase_NegControls/GC_DNase_negative_blood_shuffeled.metadata.tsv.gz',
 '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/GC_Mendelian_variants/GC_Mendelian_variants.metadata.tsv.gz',
 '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/GC_Vista/GC_Vista.metadata.tsv.gz',
 '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/GC_Cort_Chengyu/GC_Cort_Chengyu.metadata.tsv.gz',
 '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/GC_Hon/GC_Hon.metadata.tsv.gz',
 '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/C_positive_heart_MK/C_positive_heart_MK.metadata.tsv.gz',
 '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/cardiac_neuro_cava_random/cardiac_neuro_cava_random.metadata.tsv.gz',
 '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/GC_Selvarajan/GC_Selvarajan.metadata.tsv.gz',
 '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/GC_Kircher/GC_Kircher.metadata.tsv.gz',
 '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/DNase_PosControls/GC_DNase_positive_shuffeled.metadata.tsv.gz',
 '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/DNase_PosControls/GC_DNase_positive.metadata.tsv.gz',
 '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/GC_Atrial_fib/GC_Atrial_fib.metadata.tsv.gz',
 '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/neuro_controls/C_negative_neuron_MK.metadata.tsv.gz',
 '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/neuro_controls/C_positive_neuron_NP.metadata.tsv.gz',
 '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/neuro_controls/C_positive_neuron_MK.metadata.tsv.gz',
 '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/neuro_controls/C_negative_neuron_NP.metadata.tsv.gz',
 '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/neuro_controls/C_positive_neuron_CD.metadata.tsv.gz',
 '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/MK/MK.final.metadata.tsv.gz'
 ]

In [14]:
len(metadata_file_list)

29

In [15]:
# check if all these files have a header
row_count = 0
for metadata_file in metadata_file_list:
    try:
        metadata_df = pd.read_csv(metadata_file, sep='\t', compression='gzip')
        metadata_df['SPDI'][0]
        row_count += metadata_df.shape[0]
    except:
        print(metadata_file)
        print('No header')

print('all files have a header and they have a total of', row_count, 'rows')

all files have a header and they have a total of 80319 rows


In [16]:
# loop over metadata_file_list and concatenate the dataframes
# concatenate the dataframes
concatenated_metadata_df = pd.concat([pd.read_csv(metadata_file, sep='\t', compression='gzip') for metadata_file in metadata_file_list], ignore_index=True)
# use safe eval on list
# Apply the safe_eval function to the specified columns
for col in list_columns:
    concatenated_metadata_df[col] = concatenated_metadata_df[col].apply(safe_eval)

In [17]:
concatenated_metadata_df.shape[0] # 80319

80319

### investigate why the number of rows is different than expected: I have ~500 less entries than expected
- for cardiac_neuro_cava_random: same number 
- MK: 484 less (removed copy of references)
- GC_Mendelian_variants: 2 less (removed two headers because of unclear headers)
- GC_Mohlke: 1 less (removed one header because of unclear headers)

In [18]:
concatenated_metadata_df['tmp_label'] = concatenated_metadata_df['name'].apply(hf.get_label)

In [19]:
concatenated_metadata_df['tmp_label'].value_counts()

tmp_label
cardiac_neuro_cava_random            73940
MK                                    2422
C_positive_heart_AB                    909
GC_Selvarajan                          364
GC_Vista                               256
C_negative_heart_MK                    243
C_negative_neuron_MK                   234
GC_Mendelian_variants                  221
C_negative_neuron_NP                   217
GC_Kircher                             203
C_SLEA                                 200
GC_Cort_Chengyu                        185
C_positive_neuron_MK                   100
C_positive_heart_CAD                    99
C_positive_neuron_NP                    99
C_positive_heart_MK                     97
C_positive_neuron_CD                    96
GC_GABA_Chengyu                         85
GC_Glut_Chengyu                         83
GC_DNase_positive_shuffeled             55
GC_Atrial_fib                           45
GC_DNase_positive                       41
GC_Mohlke                               38
G

In [20]:
design_df['label'].value_counts()

label
cardiac_neuro_cava_random            73940
MK                                    2906
C_positive_heart_AB                    909
GC_Selvarajan                          364
GC_Vista                               256
C_negative_heart_MK                    243
C_negative_neuron_MK                   234
GC_Mendelian_variants                  223
C_negative_neuron_NP                   217
GC_Kircher                             203
C_SLEA                                 200
GC_Cort_Chengyu                        185
C_positive_neuron_MK                   100
C_positive_heart_CAD                    99
C_positive_neuron_NP                    99
C_positive_heart_MK                     97
C_positive_neuron_CD                    96
GC_GABA_Chengyu                         85
GC_Glut_Chengyu                         83
GC_DNase_positive_shuffeled             55
GC_Atrial_fib                           45
GC_DNase_positive                       41
GC_Mohlke                               39
GC_DN

In [21]:
design_df.shape[0] # 80806

80806

In [22]:
concatenated_metadata_df['sequence'].nunique() # 80215 # 80319 => 104 sequences are duplicated

80215

In [23]:
# add column: number of NA values per row
concatenated_metadata_df['na_count'] = concatenated_metadata_df.isna().sum(axis=1)


In [24]:
# Idea remove the duplicated sequences and combine them into one row in the following and concat them again to the metadata file in the end
duplicated_sequences = concatenated_metadata_df.loc[concatenated_metadata_df.duplicated(subset='sequence', keep=False)]
duplicated_sequences['tmp_label'].value_counts()
# combine the duplicates by combining the name column with ";" and take the other columns from the row with the lower number of NA values
# sort by sequence and na_count
duplicated_sequences = duplicated_sequences.sort_values(by=['sequence', 'na_count'])
duplicated_sequences['name'] = duplicated_sequences.groupby('sequence')['name'].transform(lambda x: ';'.join(x))
duplicated_sequences = duplicated_sequences.drop_duplicates(subset='sequence', keep='first')


In [25]:
duplicated_sequences['tmp_label_new'] = duplicated_sequences['name'].apply(lambda x: hf.get_label(x, check_for_multi_header=True, seperator=';'))
duplicated_sequences['tmp_label_new'].value_counts()

tmp_label_new
[GC_Glut_Chengyu, GC_GABA_Chengyu]                                       43
[C_negative_heart_MK, C_negative_neuron_MK]                              12
[GC_Mendelian_variants, GC_Mendelian_variants]                           12
[C_positive_neuron_MK, MK]                                               10
[C_negative_neuron_MK, MK]                                                6
[GC_Mohlke, GC_Selvarajan]                                                4
[C_positive_heart_MK, C_positive_neuron_MK, MK]                           2
[C_positive_heart_MK, MK]                                                 2
[C_positive_heart_CAD, GC_Selvarajan]                                     2
[GC_GABA_Chengyu, MK]                                                     2
[GC_GABA_Chengyu, C_positive_neuron_CD, MK]                               1
[C_positive_heart_MK, C_positive_neuron_MK]                               1
[C_positive_neuron_CD, MK]                                                

In [26]:
duplicated_sequences[col_sequence].nunique()

99

In [27]:
duplicated_sequences.head()

,name,sequence,category,class,source,ref,chr,start,end,strand,variant_class,variant_pos,SPDI,allele,info,tmp_label,na_count,tmp_label_new
2407,C_positive_heart_MK:tile_7938_chr11_65487464_6...,AAATGGCTGACAGCCTCTTCTGCCTTCAGAAAAGTACTTCAACTGC...,element,element inactive control,general controls IGVF year 1 design 2023,GRCh38,chr11,65487463.0,65487733.0,.,None,None,None,None,NaN,C_positive_heart_MK,5,"[C_positive_heart_MK, C_positive_neuron_MK, MK]"
77494,C_positive_neuron_MK:rdhs_106697_chr11_2249189...,AAGCACAGAAATGTTTACCTTTGTCTGATCTCTAATTCCCTAAATT...,element,element active control,Michael Kosicki,GRCh38,chr11,22491896.0,22492166.0,+,None,None,None,None,NaN,C_positive_neuron_MK,5,"[C_positive_neuron_MK, MK]"
1080,GC_Glut_Chengyu:Glut|chr7:31261174-31261443|-|...,AAGCCCCTGAGTTTTGGAGTGGTTGGTTACGCAGCAAATCTAACTG...,element,element inactive control,NaN,GRCh38,chr7,31261173.0,31261443.0,NaN,None,None,None,None,NaN,GC_Glut_Chengyu,7,"[GC_Glut_Chengyu, GC_GABA_Chengyu]"
1081,GC_Glut_Chengyu:Glut|chr7:31261189-31261458|-|...,AATGACTGTTGATTTAAGCCCCTGAGTTTTGGAGTGGTTGGTTACG...,element,element inactive control,NaN,GRCh38,chr7,31261188.0,31261458.0,NaN,None,None,None,None,NaN,GC_Glut_Chengyu,7,"[GC_Glut_Chengyu, GC_GABA_Chengyu]"
1051,GC_Glut_Chengyu:Glut|chr10:26798799-26799068|-...,AATGTAGTTAATGATATATTTGAATTCAAATGAAACAACATCATTT...,element,element inactive control,NaN,GRCh38,chr10,26798798.0,26799068.0,NaN,None,None,None,None,NaN,GC_Glut_Chengyu,7,"[GC_Glut_Chengyu, GC_GABA_Chengyu]"


### Deduplicate the final file

In [28]:
concatenated_metadata_df_deduplicated = concatenated_metadata_df.drop_duplicates(subset='sequence', keep=False)

concatenated_metadata_df_deduplicated[col_sequence].nunique()# 80116

concatenated_metadata_df_deduplicated = pd.concat([concatenated_metadata_df_deduplicated, duplicated_sequences], ignore_index=True)

In [29]:
concatenated_metadata_df_deduplicated[col_sequence].nunique() # 80215

80215

In [30]:
concatenated_metadata_df_deduplicated.shape[0]

80215

In [31]:
# make col start / col_end to int not float split into na value within col_start and col_end
contains_genomic_start = concatenated_metadata_df_deduplicated.loc[concatenated_metadata_df_deduplicated[col_start].notna()]

In [32]:
contains_genomic_start.shape[0]

77809

In [33]:
# only scrambled sequences do not have a genomic start
concatenated_metadata_df_deduplicated.loc[concatenated_metadata_df_deduplicated[col_start].isna()].tmp_label.value_counts()


tmp_label
MK                      2397
C_negative_neuron_NP       6
C_positive_neuron_CD       3
Name: count, dtype: int64

In [34]:
# make col_start, col_end to strings
concatenated_metadata_df_deduplicated[col_start] = concatenated_metadata_df_deduplicated[col_start].astype(str)
concatenated_metadata_df_deduplicated[col_end] = concatenated_metadata_df_deduplicated[col_end].astype(str)


In [35]:
import numpy as np
def remove_float_ending(genomic_coordinate):
    if "nan" in genomic_coordinate:
        return np.nan
    if "n" in genomic_coordinate.lower():
        return np.nan
    if ".0" in genomic_coordinate:
        return genomic_coordinate.split(".0")[0]
    return genomic_coordinate


concatenated_metadata_df_deduplicated[col_start] = concatenated_metadata_df_deduplicated[col_start].apply(remove_float_ending)
concatenated_metadata_df_deduplicated[col_end] = concatenated_metadata_df_deduplicated[col_end].apply(remove_float_ending)
# concatenated_metadata_df_deduplicated[col_variant_pos] = concatenated_metadata_df_deduplicated[col_variant_pos].apply(remove_float_ending)

In [36]:
# make variant pos
concatenated_metadata_df_deduplicated

,name,sequence,category,class,source,ref,chr,start,end,strand,variant_class,variant_pos,SPDI,allele,info,tmp_label,na_count,tmp_label_new
0,GC_Mohlke:REF_NC000001.11|159752292|A|G|Mohlke...,ATACATCCTTTAATTTGTTCCTACATCTTGCTTGGATTTTCCCCTG...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,159752058,159752328,+,[SNV],[234],[NC_000001.11:159752292:A:G],[ref],NaN,GC_Mohlke,1,NaN
1,GC_Mohlke:REF_NC000001.11|230158967|C|A|Mohlke...,TGTGTCTGGTGAGGTTGCTGACACTGCTTTTGGATGAGAGAGAGAG...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,230159023,230159293,+,[SNV],[145],[NC_000001.11:230159168:C:T],[ref],NaN,GC_Mohlke,1,NaN
2,GC_Mohlke:REF_NC000001.11|230158967|C|A|Mohlke...,GTTTACCCAGCCGTGGGAAAGGACGCTGTACCCCTGCCCTATTGGC...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,230159136,230159406,+,"[SNV, indel]","[32, 192]","[NC_000001.11:230159168:C:T, NC_000001.11:2301...","[ref, ref]",NaN,GC_Mohlke,1,NaN
3,GC_Mohlke:ALT_NC000001.11|230158967|C|A|Mohlke...,GTTTACCCAGCCGTGGGAAAGGACGCTGTACCCCTGCCCTATTGGC...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,230159136,230159406,+,[indel],[192],[NC_000001.11:230159328:TCTTAAAGTGTTCAGCACTCCC:T],[alt],NaN,GC_Mohlke,1,NaN
4,GC_Mohlke:REF_NC000001.11|230161389|C|T|Mohlke...,CCTCAACTCTCCACATGCCCCAGTAGCATAGACCAGCTTCCTTACA...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,230161269,230161539,+,[SNV],[120],[NC_000001.11:230161389:C:T],[ref],NaN,GC_Mohlke,1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80210,GC_Glut_Chengyu:Glut|chr7:31260964-31261233|+|...,TTTCTTATACTGTATGTTTAAAGATATAGACAATGCATATACAAGT...,element,element inactive control,NaN,GRCh38,chr7,31260963,31261233,NaN,None,None,None,None,NaN,GC_Glut_Chengyu,7,"[GC_Glut_Chengyu, GC_GABA_Chengyu]"
80211,C_negative_heart_MK:tile_8033_chr11_69157415_6...,TTTGCAGATTACGTCAGCGCTGGCGAGCAGGGGGCGCGGGGACAGC...,element,element inactive control,general controls IGVF year 1 design 2023,GRCh38,chr11,69157414,69157684,.,None,None,None,None,NaN,C_negative_heart_MK,5,"[C_negative_heart_MK, C_negative_neuron_MK]"
80212,C_negative_heart_MK:tile_22216_chr2_85772228_8...,TTTGTGCCGCCGGCGAGGACTGACACCAGGTTAAGTCGCCTGGCGG...,element,element inactive control,general controls IGVF year 1 design 2023,GRCh38,chr2,85772227,85772497,.,None,None,None,None,NaN,C_negative_heart_MK,5,"[C_negative_heart_MK, C_negative_neuron_MK]"
80213,C_positive_neuron_MK:tile_37243_chr6_97306567_...,TTTTTATTGTACTAATCAGTCTCAAGATGTTCTCTGTGTTAATTGG...,variant,variant positive control,Michael Kosicki,GRCh38,chr6,97306566,97306836,+,[SNV],[244],[NC_000006.12:97306810:G:C],[alt],NaN,C_positive_neuron_MK,1,"[C_positive_neuron_MK, MK]"


In [37]:
# check for na values in cardiac group (allele)
concatenated_metadata_df_deduplicated.columns

Index(['name', 'sequence', 'category', 'class', 'source', 'ref', 'chr',
       'start', 'end', 'strand', 'variant_class', 'variant_pos', 'SPDI',
       'allele', 'info', 'tmp_label', 'na_count', 'tmp_label_new'],
      dtype='object')

In [38]:
concatenated_metadata_df_deduplicated.dtypes

name             object
sequence         object
category         object
class            object
source           object
ref              object
chr              object
start            object
end              object
strand           object
variant_class    object
variant_pos      object
SPDI             object
allele           object
info             object
tmp_label        object
na_count          int64
tmp_label_new    object
dtype: object

In [39]:
def max_mentioned_fixes(row):
    """
    Max: 1) replace . with + in strand
    remove ";" from info column
    make an array of allele and SPDI if it is just one element
    NA in class
    """

    variant_pos = row[col_variant_pos]
    # print(type(variant_pos))
    if isinstance(variant_pos, str):
        # print(row)
        if "n" in variant_pos.lower():
            row[col_variant_pos] = np.nan
            return row
        elif "[" in variant_pos:
            return row
        else:
            if ".0" in variant_pos:
                row[col_variant_pos] = [variant_pos.split(".0")[0]]
    return row

In [40]:
# concatenated_metadata_df_deduplicated[col_variant_pos] = concatenated_metadata_df_deduplicated[col_variant_pos].astype(str)

testing_variant_pos = concatenated_metadata_df_deduplicated.apply(lambda row: max_mentioned_fixes(row), axis=1)
# concatenated_metadata_df_deduplicated = concatenated_metadata_df_deduplicated.apply(lambda row: max_mentioned_fixes(row), axis=1)


In [41]:
concatenated_metadata_df_deduplicated

,name,sequence,category,class,source,ref,chr,start,end,strand,variant_class,variant_pos,SPDI,allele,info,tmp_label,na_count,tmp_label_new
0,GC_Mohlke:REF_NC000001.11|159752292|A|G|Mohlke...,ATACATCCTTTAATTTGTTCCTACATCTTGCTTGGATTTTCCCCTG...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,159752058,159752328,+,[SNV],[234],[NC_000001.11:159752292:A:G],[ref],NaN,GC_Mohlke,1,NaN
1,GC_Mohlke:REF_NC000001.11|230158967|C|A|Mohlke...,TGTGTCTGGTGAGGTTGCTGACACTGCTTTTGGATGAGAGAGAGAG...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,230159023,230159293,+,[SNV],[145],[NC_000001.11:230159168:C:T],[ref],NaN,GC_Mohlke,1,NaN
2,GC_Mohlke:REF_NC000001.11|230158967|C|A|Mohlke...,GTTTACCCAGCCGTGGGAAAGGACGCTGTACCCCTGCCCTATTGGC...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,230159136,230159406,+,"[SNV, indel]","[32, 192]","[NC_000001.11:230159168:C:T, NC_000001.11:2301...","[ref, ref]",NaN,GC_Mohlke,1,NaN
3,GC_Mohlke:ALT_NC000001.11|230158967|C|A|Mohlke...,GTTTACCCAGCCGTGGGAAAGGACGCTGTACCCCTGCCCTATTGGC...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,230159136,230159406,+,[indel],[192],[NC_000001.11:230159328:TCTTAAAGTGTTCAGCACTCCC:T],[alt],NaN,GC_Mohlke,1,NaN
4,GC_Mohlke:REF_NC000001.11|230161389|C|T|Mohlke...,CCTCAACTCTCCACATGCCCCAGTAGCATAGACCAGCTTCCTTACA...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,230161269,230161539,+,[SNV],[120],[NC_000001.11:230161389:C:T],[ref],NaN,GC_Mohlke,1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80210,GC_Glut_Chengyu:Glut|chr7:31260964-31261233|+|...,TTTCTTATACTGTATGTTTAAAGATATAGACAATGCATATACAAGT...,element,element inactive control,NaN,GRCh38,chr7,31260963,31261233,NaN,None,None,None,None,NaN,GC_Glut_Chengyu,7,"[GC_Glut_Chengyu, GC_GABA_Chengyu]"
80211,C_negative_heart_MK:tile_8033_chr11_69157415_6...,TTTGCAGATTACGTCAGCGCTGGCGAGCAGGGGGCGCGGGGACAGC...,element,element inactive control,general controls IGVF year 1 design 2023,GRCh38,chr11,69157414,69157684,.,None,None,None,None,NaN,C_negative_heart_MK,5,"[C_negative_heart_MK, C_negative_neuron_MK]"
80212,C_negative_heart_MK:tile_22216_chr2_85772228_8...,TTTGTGCCGCCGGCGAGGACTGACACCAGGTTAAGTCGCCTGGCGG...,element,element inactive control,general controls IGVF year 1 design 2023,GRCh38,chr2,85772227,85772497,.,None,None,None,None,NaN,C_negative_heart_MK,5,"[C_negative_heart_MK, C_negative_neuron_MK]"
80213,C_positive_neuron_MK:tile_37243_chr6_97306567_...,TTTTTATTGTACTAATCAGTCTCAAGATGTTCTCTGTGTTAATTGG...,variant,variant positive control,Michael Kosicki,GRCh38,chr6,97306566,97306836,+,[SNV],[244],[NC_000006.12:97306810:G:C],[alt],NaN,C_positive_neuron_MK,1,"[C_positive_neuron_MK, MK]"


#### Sanity checks because of the variant position which was set to NA for reference sequences
- this was then changed to the same handling as the SPDI and allele column

In [42]:
(testing_variant_pos[col_variant_pos].isna()).sum()

14123

In [43]:
testing_variant_pos[col_variant_pos].isna().sum()
testing_variant_pos[col_allele].str.contains("ref").sum()

0.0

In [44]:
testing_variant_pos.loc[(testing_variant_pos[col_variant_class] == "SNV") & (testing_variant_pos[col_variant_pos].isna())]['tmp_label'].value_counts()
testing_variant_pos.loc[(testing_variant_pos[col_variant_class] == "SNV") & (testing_variant_pos[col_SPDI].isna()) & (testing_variant_pos[col_allele].str.contains("ref")) & (testing_variant_pos[col_variant_pos].isna())]['tmp_label'].value_counts()
testing_variant_pos.loc[(testing_variant_pos[col_variant_class] == "SNV") & (testing_variant_pos[col_variant_pos].isna())]['tmp_label'].value_counts()
snv_no_variant_pos = testing_variant_pos.loc[(testing_variant_pos[col_variant_class] == "SNV") & (testing_variant_pos[col_variant_pos].isna())]
# testing_variant_pos.loc[(testing_variant_pos[col_variant_class] == "SNV") & (testing_variant_pos[col_SPDI].isna()) & (testing_variant_pos[col_allele].str.contains("ref")) & (testing_variant_pos[col_variant_pos].isna())]

In [45]:
snv_no_variant_pos['tmp_label'].value_counts()

Series([], Name: count, dtype: int64)

In [46]:
snv_no_variant_pos

,name,sequence,category,class,source,ref,chr,start,end,strand,variant_class,variant_pos,SPDI,allele,info,tmp_label,na_count,tmp_label_new


In [47]:
testing_variant_pos.dtypes

name             object
sequence         object
category         object
class            object
source           object
ref              object
chr              object
start            object
end              object
strand           object
variant_class    object
variant_pos      object
SPDI             object
allele           object
info             object
tmp_label        object
na_count          int64
tmp_label_new    object
dtype: object

In [48]:
concatenated_metadata_df_deduplicated[col_variant_class]

0               [SNV]
1               [SNV]
2        [SNV, indel]
3             [indel]
4               [SNV]
             ...     
80210            None
80211            None
80212            None
80213           [SNV]
80214            None
Name: variant_class, Length: 80215, dtype: object

In [49]:
concatenated_metadata_df_deduplicated[col_variant_class].value_counts()

variant_class
[SNV]                                                                                                                                                                                                                                                                                                                                                                                                               55494
[SNV, SNV]                                                                                                                                                                                                                                                                                                                                                                                                           4808
[SNV, SNV, SNV]                                                                                                                                                       

In [43]:
stopping here

SyntaxError: invalid syntax (2707631692.py, line 1)

In [50]:
concatenated_metadata_df_deduplicated

,name,sequence,category,class,source,ref,chr,start,end,strand,variant_class,variant_pos,SPDI,allele,info,tmp_label,na_count,tmp_label_new
0,GC_Mohlke:REF_NC000001.11|159752292|A|G|Mohlke...,ATACATCCTTTAATTTGTTCCTACATCTTGCTTGGATTTTCCCCTG...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,159752058,159752328,+,[SNV],[234],[NC_000001.11:159752292:A:G],[ref],NaN,GC_Mohlke,1,NaN
1,GC_Mohlke:REF_NC000001.11|230158967|C|A|Mohlke...,TGTGTCTGGTGAGGTTGCTGACACTGCTTTTGGATGAGAGAGAGAG...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,230159023,230159293,+,[SNV],[145],[NC_000001.11:230159168:C:T],[ref],NaN,GC_Mohlke,1,NaN
2,GC_Mohlke:REF_NC000001.11|230158967|C|A|Mohlke...,GTTTACCCAGCCGTGGGAAAGGACGCTGTACCCCTGCCCTATTGGC...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,230159136,230159406,+,"[SNV, indel]","[32, 192]","[NC_000001.11:230159168:C:T, NC_000001.11:2301...","[ref, ref]",NaN,GC_Mohlke,1,NaN
3,GC_Mohlke:ALT_NC000001.11|230158967|C|A|Mohlke...,GTTTACCCAGCCGTGGGAAAGGACGCTGTACCCCTGCCCTATTGGC...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,230159136,230159406,+,[indel],[192],[NC_000001.11:230159328:TCTTAAAGTGTTCAGCACTCCC:T],[alt],NaN,GC_Mohlke,1,NaN
4,GC_Mohlke:REF_NC000001.11|230161389|C|T|Mohlke...,CCTCAACTCTCCACATGCCCCAGTAGCATAGACCAGCTTCCTTACA...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,230161269,230161539,+,[SNV],[120],[NC_000001.11:230161389:C:T],[ref],NaN,GC_Mohlke,1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80210,GC_Glut_Chengyu:Glut|chr7:31260964-31261233|+|...,TTTCTTATACTGTATGTTTAAAGATATAGACAATGCATATACAAGT...,element,element inactive control,NaN,GRCh38,chr7,31260963,31261233,NaN,None,None,None,None,NaN,GC_Glut_Chengyu,7,"[GC_Glut_Chengyu, GC_GABA_Chengyu]"
80211,C_negative_heart_MK:tile_8033_chr11_69157415_6...,TTTGCAGATTACGTCAGCGCTGGCGAGCAGGGGGCGCGGGGACAGC...,element,element inactive control,general controls IGVF year 1 design 2023,GRCh38,chr11,69157414,69157684,.,None,None,None,None,NaN,C_negative_heart_MK,5,"[C_negative_heart_MK, C_negative_neuron_MK]"
80212,C_negative_heart_MK:tile_22216_chr2_85772228_8...,TTTGTGCCGCCGGCGAGGACTGACACCAGGTTAAGTCGCCTGGCGG...,element,element inactive control,general controls IGVF year 1 design 2023,GRCh38,chr2,85772227,85772497,.,None,None,None,None,NaN,C_negative_heart_MK,5,"[C_negative_heart_MK, C_negative_neuron_MK]"
80213,C_positive_neuron_MK:tile_37243_chr6_97306567_...,TTTTTATTGTACTAATCAGTCTCAAGATGTTCTCTGTGTTAATTGG...,variant,variant positive control,Michael Kosicki,GRCh38,chr6,97306566,97306836,+,[SNV],[244],[NC_000006.12:97306810:G:C],[alt],NaN,C_positive_neuron_MK,1,"[C_positive_neuron_MK, MK]"


In [51]:
def max_mentioned_fixes(row):
    """
    Max: 1) replace . with + in strand
    remove ";" from info column
    make an array of allele and SPDI if it is just one element
    NA in class
    """
    strand = row[col_strand]
    info = row[col_info]
    allele = row[col_allele]
    SPDI = row[col_SPDI]
    variant_class = row[col_variant_class]
    # print(type(allele))
    # print(allele)
    # print(type(SPDI))
    # print(SPDI)
    if allele == 'ref' or allele == 'alt': # only "alt" is string
        row[col_allele] = [allele]
    if isinstance(SPDI, str): # only for alt sequences this is true
        if SPDI[0] != "[":
            row[col_SPDI] = [SPDI]
    if isinstance(variant_class, str):
        if row[col_variant_class] in ['SNV', 'indel']:
                row[col_variant_class] = [variant_class]
    if isinstance(strand, str):
        if "." in strand:
            row[col_strand] = "+"
    if isinstance(info, str):
        if ";" in info:
            row[col_info] = ""
    if not isinstance(row[col_class], str):
        print(row[col_name])
    return row
test_of_fixed_metadata = concatenated_metadata_df_deduplicated.apply(lambda row: max_mentioned_fixes(row), axis=1)
concatenated_metadata_df_deduplicated = concatenated_metadata_df_deduplicated.apply(lambda row: max_mentioned_fixes(row), axis=1)
# info column: remove: strand information not restorable from header;

In [52]:
# Data with no strand information
concatenated_metadata_df_deduplicated.loc[concatenated_metadata_df_deduplicated['strand'].isna()]['tmp_label'].value_counts()

tmp_label
MK                      2397
GC_Vista                 256
GC_Cort_Chengyu          185
GC_Glut_Chengyu           83
GC_GABA_Chengyu           42
GC_Hon                     6
C_negative_neuron_NP       6
C_positive_neuron_CD       3
Name: count, dtype: int64

In [53]:
concatenated_metadata_df_deduplicated.columns

Index(['name', 'sequence', 'category', 'class', 'source', 'ref', 'chr',
       'start', 'end', 'strand', 'variant_class', 'variant_pos', 'SPDI',
       'allele', 'info', 'tmp_label', 'na_count', 'tmp_label_new'],
      dtype='object')

In [54]:
# split the metadata
chengyu_concatenated_metadata_df_deduplicated = concatenated_metadata_df_deduplicated.loc[concatenated_metadata_df_deduplicated['tmp_label'].isin(["GC_Glut_Chengyu", "GC_GABA_Chengyu"])].copy()
concatenated_metadata_df_deduplicated_no_chengyu = concatenated_metadata_df_deduplicated.loc[~concatenated_metadata_df_deduplicated['tmp_label'].isin(["GC_Glut_Chengyu", "GC_GABA_Chengyu"])].copy()

In [55]:
minus_strand_chengyu_seqs = "/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_2025_design_WTC11/results/80k_controls/test_minusstrand_GC_Chengyu_controls.bed"

minus_strand_chengyu_seqs_df = pd.read_csv(minus_strand_chengyu_seqs, sep="\t", header=None, names=['chr', 'start', 'end', 'name', 'score', 'strand'])
minus_strand_chengyu_seqs_df['label'] = minus_strand_chengyu_seqs_df['name'].apply(hf.get_label)
print(minus_strand_chengyu_seqs_df['label'].value_counts())

def add_stand_chengyu_controls(row, name_column = "name", col_strand='strand'):
    row[col_strand] = '+'
    if row[name_column] in minus_strand_chengyu_seqs_df['name'].to_list():
        row[col_strand] = '-'
    return row

chengyu_concatenated_metadata_df_deduplicated = chengyu_concatenated_metadata_df_deduplicated.apply(lambda row: add_stand_chengyu_controls(row, name_column=col_name), axis=1)

label
GC_Glut_Chengyu    39
GC_GABA_Chengyu    17
Name: count, dtype: int64


In [56]:
chengyu_concatenated_metadata_df_deduplicated

,name,sequence,category,class,source,ref,chr,start,end,strand,variant_class,variant_pos,SPDI,allele,info,tmp_label,na_count,tmp_label_new
1040,GC_Glut_Chengyu:Glut|chr10:26004717-26004986|+...,ACTTGAAGCAATTCTTACTCTCTAGATCTGGGACAGTTTGAGCACC...,element,element inactive control,NaN,GRCh38,chr10,26004716,26004986,+,None,None,None,None,NaN,GC_Glut_Chengyu,7,NaN
1041,GC_Glut_Chengyu:Glut|chr10:26004732-26005001|+...,TACTCTCTAGATCTGGGACAGTTTGAGCACCAAAATACTAATATTG...,element,element inactive control,NaN,GRCh38,chr10,26004731,26005001,+,None,None,None,None,NaN,GC_Glut_Chengyu,7,NaN
1042,GC_Glut_Chengyu:Glut|chr10:26004747-26005016|+...,GGACAGTTTGAGCACCAAAATACTAATATTGCTGGTAATGATAGTA...,element,element inactive control,NaN,GRCh38,chr10,26004746,26005016,+,None,None,None,None,NaN,GC_Glut_Chengyu,7,NaN
1043,GC_Glut_Chengyu:Glut|chr10:26004747-26005016|-...,TCAAAACTTAACTCACTAGTTTTAGCAGCCATTGAAATTTTTGGAT...,element,element inactive control,NaN,GRCh38,chr10,26004746,26005016,-,None,None,None,None,NaN,GC_Glut_Chengyu,7,NaN
1044,GC_Glut_Chengyu:Glut|chr10:26004762-26005031|+...,CAAAATACTAATATTGCTGGTAATGATAGTAATTCACTGAATGCAA...,element,element inactive control,NaN,GRCh38,chr10,26004761,26005031,+,None,None,None,None,NaN,GC_Glut_Chengyu,7,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80203,GC_Glut_Chengyu:Glut|chr12:44940899-44941168|-...,TGTCCAAAAAAAGTAAAAGTCATAACAGAAATTGGATTTCAAATGG...,element,element inactive control,NaN,GRCh38,chr12,44940898,44941168,-,None,None,None,None,NaN,GC_Glut_Chengyu,7,"[GC_Glut_Chengyu, GC_GABA_Chengyu]"
80204,GC_Glut_Chengyu:Glut|chr2:212699381-212699650|...,TGTGAGTGCTATAATTGTAACCCTTTATAATTGACAGACATTAATT...,element,element inactive control,NaN,GRCh38,chr2,212699380,212699650,+,None,None,None,None,NaN,GC_Glut_Chengyu,7,"[GC_Glut_Chengyu, GC_GABA_Chengyu]"
80209,GC_Glut_Chengyu:Glut|chr10:26798829-26799098|-...,TTTAATAGGAGTACTATTGAATTTACATTTAATGTAGTTAATGATA...,element,element inactive control,NaN,GRCh38,chr10,26798828,26799098,-,None,None,None,None,NaN,GC_Glut_Chengyu,7,"[GC_Glut_Chengyu, GC_GABA_Chengyu]"
80210,GC_Glut_Chengyu:Glut|chr7:31260964-31261233|+|...,TTTCTTATACTGTATGTTTAAAGATATAGACAATGCATATACAAGT...,element,element inactive control,NaN,GRCh38,chr7,31260963,31261233,+,None,None,None,None,NaN,GC_Glut_Chengyu,7,"[GC_Glut_Chengyu, GC_GABA_Chengyu]"


In [57]:
concatenated_metadata_df = pd.concat([concatenated_metadata_df_deduplicated_no_chengyu, chengyu_concatenated_metadata_df_deduplicated], ignore_index=True)

In [58]:
concatenated_metadata_df.sample(10)

,name,sequence,category,class,source,ref,chr,start,end,strand,variant_class,variant_pos,SPDI,allele,info,tmp_label,na_count,tmp_label_new
58769,cardiac_neuro_cava_random:ALT_IGLV3-25|ENSG000...,CTGCCTAGCTCCTGGTGCCAAGAATGATGAACCCTCTGCATTGCCC...,variant,test,NaN,GRCh38,chr22,22655951,22656221,+,[SNV],[124],[NC_000022.11:22656075:C:T],[alt],NaN,cardiac_neuro_cava_random,2,NaN
64047,cardiac_neuro_cava_random:ALT_HAND2|ENSG000001...,ACCCCATACTAATTAAAGCAGACTCTCTCAGGGTGGGGCCTCAGAC...,variant,test,NaN,GRCh38,chr4,173486833,173487103,-,[SNV],[244],[NC_000004.12:173486858:G:A],[alt],NaN,cardiac_neuro_cava_random,2,NaN
36426,cardiac_neuro_cava_random:ALT_CTNNA3|ENSG00000...,ACAAAAAGACCTTAATTCTGGAGGCAAACTTCAACTTCATCCAGCA...,variant,test,NaN,GRCh38,chr10,67485994,67486264,-,[SNV],[188],[NC_000010.11:67486075:T:C],[alt],NaN,cardiac_neuro_cava_random,2,NaN
23821,cardiac_neuro_cava_random:REF_CACNA2D3|ENSG000...,TTATGATATGATAGTATAAAAAATTAAAGTCACCTGCCTCCAAAAA...,variant,test,NaN,GRCh38,chr3,54897384,54897654,+,[SNV],[150],[NC_000003.12:54897534:T:C],[ref],NaN,cardiac_neuro_cava_random,2,NaN
57457,cardiac_neuro_cava_random:ALT_JPH2|ENSG0000014...,GAGCCATGTAGGTATCTGGAGGAAGGGCATTACAGGTAGAGGGAAC...,variant,test,NaN,GRCh38,chr20,44165534,44165804,-,[SNV],[174],[NC_000020.11:44165629:C:T],[alt],NaN,cardiac_neuro_cava_random,2,NaN
18109,cardiac_neuro_cava_random:REF_GRIN2A|ENSG00000...,TGTCTTAGATGCAAGAGTCAGACTGACGGAGGGCTCCTTTCTAGGT...,variant,test,NaN,GRCh38,chr16,10087238,10087508,-,[SNV],[117],[NC_000016.10:10087390:C:G],[ref],NaN,cardiac_neuro_cava_random,2,NaN
13351,cardiac_neuro_cava_random:REF_DISC1|ENSG000001...,CAGGGCTCCCTGGCGGGGGGCACTGATTATGCAGGGTGCTGCAGGC...,variant,test,NaN,GRCh38,chr1,231776593,231776863,+,[SNV],[123],[NC_000001.11:231776716:A:C],[ref],NaN,cardiac_neuro_cava_random,2,NaN
32705,cardiac_neuro_cava_random:ALT_CNN3|ENSG0000011...,AGATTGTTTCATATTTCCTGCCATAGCTATAAATTGACTTTAGGTT...,variant,test,NaN,GRCh38,chr1,94855579,94855849,-,[SNV],[44],[NC_000001.11:94855804:G:A],[alt],NaN,cardiac_neuro_cava_random,2,NaN
73088,cardiac_neuro_cava_random:ALT_FOXH1|ENSG000001...,CTCCCACAGGACAGCTGCTTCAATAGCATCTGTGCCCCCTCCTGCC...,variant,test,NaN,GRCh38,chr8,144455249,144455519,-,[SNV],[84],[NC_000008.11:144455434:C:T],[alt],NaN,cardiac_neuro_cava_random,2,NaN
18477,cardiac_neuro_cava_random:REF_CTCF|ENSG0000010...,ATGGTGAAACCCCATGTTGTCTCTACTTAAAAAATAAAAATCGCCC...,variant,test,NaN,GRCh38,chr16,67549989,67550259,+,"[SNV, SNV]","[201, 223]","[NC_000016.10:67550190:G:A, NC_000016.10:67550...","[ref, ref]",NaN,cardiac_neuro_cava_random,2,NaN


In [59]:
test_of_fixed_metadata.loc[test_of_fixed_metadata[col_allele].apply(lambda allele: allele == ["NA"])]['tmp_label'].value_counts()

Series([], Name: count, dtype: int64)

In [60]:
concatenated_metadata_df[col_class].value_counts()
variant_positive = concatenated_metadata_df.loc[concatenated_metadata_df[col_class] == "variant positive control"]
element_positive = concatenated_metadata_df.loc[concatenated_metadata_df[col_class] == "element active control"]

In [61]:
concatenated_metadata_df.loc[concatenated_metadata_df[col_variant_class].isin(['SNV', 'indel'])].apply(lambda row: row['name'].split(":")[0], axis=1).value_counts()

Series([], Name: count, dtype: int64)

In [62]:
print(variant_positive.shape[0])
variant_positive
element_positive

372


,name,sequence,category,class,source,ref,chr,start,end,strand,variant_class,variant_pos,SPDI,allele,info,tmp_label,na_count,tmp_label_new
77145,C_positive_neuron_NP:GW18_PFC_ABC_chr5_6873074...,CTTTAAAAATGCAGATAGGAATATCAGACCTAGAAGTTTTCTGTGC...,element,element active control,Nick Page,GRCh38,chr5,68730744,68731014,+,None,None,None,None,NaN,C_positive_neuron_NP,5,NaN
77146,C_positive_neuron_NP:GW18_PFC_ABC_chr5_6873083...,CTTTGTAACGGAAATTGGCTCTGATTTCTGAGATGGGCATTACATC...,element,element active control,Nick Page,GRCh38,chr5,68730834,68731104,+,None,None,None,None,NaN,C_positive_neuron_NP,5,NaN
77147,C_positive_neuron_NP:NGN2_iPSC_ABC_chrX_743252...,TGCCCCAGAGATCAGCTTTTCAAGAACAGGCATCTGAGACAGAGTG...,element,element active control,Nick Page,GRCh38,chrX,74325294,74325564,+,None,None,None,None,NaN,C_positive_neuron_NP,5,NaN
77148,C_positive_neuron_NP:GW18_PFC_ABC_NGN2_iPSC_AB...,CAAGTTGAAAATGTGACAAAGCTCAATTTCATTTCTTGTTTATTGC...,element,element active control,Nick Page,GRCh38,chr21,37435137,37435407,+,None,None,None,None,NaN,C_positive_neuron_NP,5,NaN
77149,C_positive_neuron_NP:NGN2_iPSC_ABC_chr10_88387...,CCGGAAGCCTTAAGTATTTGGAACTGCTCCAAACCCAGAAGAGGGG...,element,element active control,Nick Page,GRCh38,chr10,88387160,88387430,+,None,None,None,None,NaN,C_positive_neuron_NP,5,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80050,C_positive_neuron_MK:tile_23593_chr2_174331518...,CAGAGCAACTTGAGCAGTTGCTCAGATCAAAGCTCCCGGGAGAGGC...,element,element active control,Michael Kosicki,GRCh38,chr2,174331517,174331787,+,None,None,None,None,NaN,C_positive_neuron_MK,5,"[C_positive_neuron_MK, MK]"
80054,C_positive_neuron_MK:tile_45129_chr9_16710917_...,CATTCAATATGGTCAAGTCCTTAATTGGCTGGGCTTCCTGAGAAAG...,element,element active control,Michael Kosicki,GRCh38,chr9,16710916,16711186,+,None,None,None,None,NaN,C_positive_neuron_MK,5,"[C_positive_neuron_MK, MK]"
80070,C_positive_neuron_MK:tile_34838_chr5_142063877...,GCATTGAAAGGCAGCTGTAATTAGAAAGCATAGGCATTCAAAGCTG...,element,element active control,Michael Kosicki,GRCh38,chr5,142063876,142064146,+,None,None,None,None,NaN,C_positive_neuron_MK,5,"[C_positive_neuron_MK, MK]"
80074,C_positive_neuron_MK:tile_31993_chr4_139816289...,GGTGTGAGTTCAAAAACAAACAAGTGAAAGTGCCGAGGGATAAAAT...,element,element active control,Michael Kosicki,GRCh38,chr4,139816288,139816558,+,None,None,None,None,NaN,C_positive_neuron_MK,5,"[C_positive_neuron_MK, MK]"


In [63]:
concatenated_metadata_df[col_variant_pos].isna().sum() # 14123

14123

In [64]:
# TODO: get the groups with SNV on - strand
concatenated_metadata_df

,name,sequence,category,class,source,ref,chr,start,end,strand,variant_class,variant_pos,SPDI,allele,info,tmp_label,na_count,tmp_label_new
0,GC_Mohlke:REF_NC000001.11|159752292|A|G|Mohlke...,ATACATCCTTTAATTTGTTCCTACATCTTGCTTGGATTTTCCCCTG...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,159752058,159752328,+,[SNV],[234],[NC_000001.11:159752292:A:G],[ref],NaN,GC_Mohlke,1,NaN
1,GC_Mohlke:REF_NC000001.11|230158967|C|A|Mohlke...,TGTGTCTGGTGAGGTTGCTGACACTGCTTTTGGATGAGAGAGAGAG...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,230159023,230159293,+,[SNV],[145],[NC_000001.11:230159168:C:T],[ref],NaN,GC_Mohlke,1,NaN
2,GC_Mohlke:REF_NC000001.11|230158967|C|A|Mohlke...,GTTTACCCAGCCGTGGGAAAGGACGCTGTACCCCTGCCCTATTGGC...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,230159136,230159406,+,"[SNV, indel]","[32, 192]","[NC_000001.11:230159168:C:T, NC_000001.11:2301...","[ref, ref]",NaN,GC_Mohlke,1,NaN
3,GC_Mohlke:ALT_NC000001.11|230158967|C|A|Mohlke...,GTTTACCCAGCCGTGGGAAAGGACGCTGTACCCCTGCCCTATTGGC...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,230159136,230159406,+,[indel],[192],[NC_000001.11:230159328:TCTTAAAGTGTTCAGCACTCCC:T],[alt],NaN,GC_Mohlke,1,NaN
4,GC_Mohlke:REF_NC000001.11|230161389|C|T|Mohlke...,CCTCAACTCTCCACATGCCCCAGTAGCATAGACCAGCTTCCTTACA...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,230161269,230161539,+,[SNV],[120],[NC_000001.11:230161389:C:T],[ref],NaN,GC_Mohlke,1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80210,GC_Glut_Chengyu:Glut|chr12:44940899-44941168|-...,TGTCCAAAAAAAGTAAAAGTCATAACAGAAATTGGATTTCAAATGG...,element,element inactive control,NaN,GRCh38,chr12,44940898,44941168,-,None,None,None,None,NaN,GC_Glut_Chengyu,7,"[GC_Glut_Chengyu, GC_GABA_Chengyu]"
80211,GC_Glut_Chengyu:Glut|chr2:212699381-212699650|...,TGTGAGTGCTATAATTGTAACCCTTTATAATTGACAGACATTAATT...,element,element inactive control,NaN,GRCh38,chr2,212699380,212699650,+,None,None,None,None,NaN,GC_Glut_Chengyu,7,"[GC_Glut_Chengyu, GC_GABA_Chengyu]"
80212,GC_Glut_Chengyu:Glut|chr10:26798829-26799098|-...,TTTAATAGGAGTACTATTGAATTTACATTTAATGTAGTTAATGATA...,element,element inactive control,NaN,GRCh38,chr10,26798828,26799098,-,None,None,None,None,NaN,GC_Glut_Chengyu,7,"[GC_Glut_Chengyu, GC_GABA_Chengyu]"
80213,GC_Glut_Chengyu:Glut|chr7:31260964-31261233|+|...,TTTCTTATACTGTATGTTTAAAGATATAGACAATGCATATACAAGT...,element,element inactive control,NaN,GRCh38,chr7,31260963,31261233,+,None,None,None,None,NaN,GC_Glut_Chengyu,7,"[GC_Glut_Chengyu, GC_GABA_Chengyu]"


In [65]:
concatenated_metadata_df

,name,sequence,category,class,source,ref,chr,start,end,strand,variant_class,variant_pos,SPDI,allele,info,tmp_label,na_count,tmp_label_new
0,GC_Mohlke:REF_NC000001.11|159752292|A|G|Mohlke...,ATACATCCTTTAATTTGTTCCTACATCTTGCTTGGATTTTCCCCTG...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,159752058,159752328,+,[SNV],[234],[NC_000001.11:159752292:A:G],[ref],NaN,GC_Mohlke,1,NaN
1,GC_Mohlke:REF_NC000001.11|230158967|C|A|Mohlke...,TGTGTCTGGTGAGGTTGCTGACACTGCTTTTGGATGAGAGAGAGAG...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,230159023,230159293,+,[SNV],[145],[NC_000001.11:230159168:C:T],[ref],NaN,GC_Mohlke,1,NaN
2,GC_Mohlke:REF_NC000001.11|230158967|C|A|Mohlke...,GTTTACCCAGCCGTGGGAAAGGACGCTGTACCCCTGCCCTATTGGC...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,230159136,230159406,+,"[SNV, indel]","[32, 192]","[NC_000001.11:230159168:C:T, NC_000001.11:2301...","[ref, ref]",NaN,GC_Mohlke,1,NaN
3,GC_Mohlke:ALT_NC000001.11|230158967|C|A|Mohlke...,GTTTACCCAGCCGTGGGAAAGGACGCTGTACCCCTGCCCTATTGGC...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,230159136,230159406,+,[indel],[192],[NC_000001.11:230159328:TCTTAAAGTGTTCAGCACTCCC:T],[alt],NaN,GC_Mohlke,1,NaN
4,GC_Mohlke:REF_NC000001.11|230161389|C|T|Mohlke...,CCTCAACTCTCCACATGCCCCAGTAGCATAGACCAGCTTCCTTACA...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,230161269,230161539,+,[SNV],[120],[NC_000001.11:230161389:C:T],[ref],NaN,GC_Mohlke,1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80210,GC_Glut_Chengyu:Glut|chr12:44940899-44941168|-...,TGTCCAAAAAAAGTAAAAGTCATAACAGAAATTGGATTTCAAATGG...,element,element inactive control,NaN,GRCh38,chr12,44940898,44941168,-,None,None,None,None,NaN,GC_Glut_Chengyu,7,"[GC_Glut_Chengyu, GC_GABA_Chengyu]"
80211,GC_Glut_Chengyu:Glut|chr2:212699381-212699650|...,TGTGAGTGCTATAATTGTAACCCTTTATAATTGACAGACATTAATT...,element,element inactive control,NaN,GRCh38,chr2,212699380,212699650,+,None,None,None,None,NaN,GC_Glut_Chengyu,7,"[GC_Glut_Chengyu, GC_GABA_Chengyu]"
80212,GC_Glut_Chengyu:Glut|chr10:26798829-26799098|-...,TTTAATAGGAGTACTATTGAATTTACATTTAATGTAGTTAATGATA...,element,element inactive control,NaN,GRCh38,chr10,26798828,26799098,-,None,None,None,None,NaN,GC_Glut_Chengyu,7,"[GC_Glut_Chengyu, GC_GABA_Chengyu]"
80213,GC_Glut_Chengyu:Glut|chr7:31260964-31261233|+|...,TTTCTTATACTGTATGTTTAAAGATATAGACAATGCATATACAAGT...,element,element inactive control,NaN,GRCh38,chr7,31260963,31261233,+,None,None,None,None,NaN,GC_Glut_Chengyu,7,"[GC_Glut_Chengyu, GC_GABA_Chengyu]"


In [ ]:
asdf # meant as breakpoint before writing

NameError: name 'asdf' is not defined

In [66]:
interesting_columns = [col_name, col_sequence, col_category,
                       col_class, col_source, col_ref, col_chr,
                       col_start, col_end, col_strand, col_variant_class,
                       col_variant_pos, col_SPDI, col_allele, col_info]
group_name = 'MPRA_80215'
output_dir = "/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format"
tmp_output_path = os.path.join(output_dir, f'{group_name}.metadata.tmp.tsv.gz')
concatenated_metadata_df[interesting_columns].to_csv(tmp_output_path, sep='\t', index=False, na_rep='NA', compression='gzip')
os.system(f'zcat {output_dir}/{group_name}.metadata.tmp.tsv.gz | sed "s/\'/\\"/g" | gzip -c > {output_dir}/{group_name}.metadata.tsv.gz')
print("Saved the metadata file to: ", f'{output_dir}/{group_name}.metadata.tsv.gz')

Saved the metadata file to:  /data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/MPRA_80215.metadata.tsv.gz


In [67]:
f'{output_dir}/{group_name}.metadata.tsv.gz'

'/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/MPRA_80215.metadata.tsv.gz'

In [68]:
os.system(f'zcat {output_dir}/{group_name}.metadata.tmp.tsv.gz | sed "s/\'/\\"/g" | gzip -c > /home/kisa/coding/80K_MPRA/80K-Analysis/07_quality_control/notebooks/control_metadata/{group_name}.metadata.tsv.gz')

0

In [ ]:
concatenated_metadata_df

#### General about sequences which are not 270bp long: 

In [ ]:
# # these sequences should be updates
# C_positive_heart_CAD:ALT_rs34091558_rs34091558
# TCTTCTCGGCCAATGAAGGGTCAACTCCATTGCTCTCAGCAGAACTAAATGCTTTGCCAACTGGTCTGGTGACTCATACTGGGAAGCTTGTAGCAAACCCCATTTTTGGAGAAACCGGGAAATCTCTTTGGGGAGATAAAACAAGTCACTGTAGCACTCTGCTCTCTGAAGTGCCTGGCTAGTACTTTCTGCTTCTTCATTTTATCCCAGCACCAAGGAAAACCCAAGCAAGTCTGGGAAATGTTGTAAAGATCTACACAAATGCAAGCT
# GC_Mendelian_variants:ALT_chr1:209816133C>CA|IRF6_chr1:209816133C>CA|IRF6
# ATTGAGCCCAGGGGCTGAATCTGGAGCTTTGGGGCCTGGGAACCTCTCTACCTGCGTCAATGTCTGGAGGCCCTGAGAGTTTCGCTCAGGCTCAGAGCAGGCATCGCAACCTCCCAGTTACTATTCTGTGCTGTGGCAAAGTGCCAGCTTGTCCTCTCTTCCCCACCCAGCCCGGGAAACCGGCAGCATTTCTAGTTCAGGCCCAGACCCGTCCTGGCAGCCTGGATTCCACTGCCTAGGCAGGAAGCTCATCTCAGCCCAGTGACCTTT

# GC_Mendelian_variants:ALT_chr7:156791255G>C|SHH_chr7:156791274T>TTAAGGAAGTGATT|SHH
# TGAGATATGGCTTCATTTTCTGTAATAAACACTAAGATCAAAACATGACCCAAGTTAAATTTCCTTGCAGGGTTCCCAGCAGGGGCTTCCCTTTTGTCTGTGATTTCCTCTCACCCACCAGAACCAGGCCAAATATGCGCATGTGCCACTAACACTTAAGGAAGTGATTAAGCAGCACTTCCTTAATCACTCATTTCCAACAATTTATGGATCATCAGTGGCAAAAAACGAGCAAAAATAATGAAAGAATGCAATGAAAGCTCGTGGAGA

# GC_Mendelian_variants:ALT_chr7:156791257G>A|SHH_chr7:156791274T>TTAAGGAAGTGATT|SHH
# AGATATGGCTTCATTTTCTGTAATAAACACTAAGATCAAAACATGACCCAAGTTAAATTTCCTTGCAGGGTTCCCAGCAGGGGCTTCCCTTTTGTCTGTGATTTCCTCTCACCCACCAGAACCAGGCCAAATATGCGCATGTGCCACTAACACTTAAGGAAGTGATTAAGCAGCACTTCCTTAATCACTCATTTCCAACAATTTATGGATCATCAGTGGCAAAAAACGAGCAAAAATAATGAAAGAATGCAATGAAAGCTCGTGGAGACA

# GC_Mendelian_variants:ALT_chr7:156791274T>TTAAGGAAGTGATT|SHH_chr7:156791274T>TTAAGGAAGTGATT|SHH
# CTGTAATAAACACTAAGATCAAAACATGACCCAAGTTAAATTTCCTTGCAGGGTTCCCAGCAGGGGCTTCCCTTTTGTCTGTGATTTCCTCTCACCCACCAGAACCAGGCCAAATATGCGCATGTGCCACTAACACTTAAGGAAGTGATTAAGCAGCACTTCCTTAATCACTCATTTCCAACAATTTATGGATCATCAGTGGCAAAAAACGAGCAAAAATAATGAAAGAATGCAATGAAAGCTCGTGGAGACAGAGGCTGGACTTCCTAC


In [ ]:
# import ast

# # Function to safely evaluate string representations of lists
# def safe_eval(x):
#     if pd.isna(x):
#         return None
#     try:
#         return ast.literal_eval(x)
#     except (ValueError, SyntaxError):
#         return x

# # list columns col_variant_class, col_variant_pos, col_SPDI, col_allele,
# list_columns = [col_variant_class, col_variant_pos, col_SPDI, col_allele]
# # Apply the safe_eval function to the specified columns
# for col in list_columns:
#     concatenated_metadata_df_deduplicated[col] = concatenated_metadata_df_deduplicated[col].apply(safe_eval)

In [ ]:
concatenated_metadata_df_deduplicated

,name,sequence,category,class,source,ref,chr,start,end,strand,variant_class,variant_pos,SPDI,allele,info,tmp_label,na_count,tmp_label_new
0,GC_Mohlke:REF_NC000001.11|159752292|A|G|Mohlke...,ATACATCCTTTAATTTGTTCCTACATCTTGCTTGGATTTTCCCCTG...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,159752058,159752328,+,[SNV],[234],[NC_000001.11:159752292:A:G],[ref],NaN,GC_Mohlke,1,NaN
1,GC_Mohlke:REF_NC000001.11|230158967|C|A|Mohlke...,TGTGTCTGGTGAGGTTGCTGACACTGCTTTTGGATGAGAGAGAGAG...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,230159023,230159293,+,[SNV],[145],[NC_000001.11:230159168:C:T],[ref],NaN,GC_Mohlke,1,NaN
2,GC_Mohlke:REF_NC000001.11|230158967|C|A|Mohlke...,GTTTACCCAGCCGTGGGAAAGGACGCTGTACCCCTGCCCTATTGGC...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,230159136,230159406,+,"[SNV, indel]","[32, 192]","[NC_000001.11:230159168:C:T, NC_000001.11:2301...","[ref, ref]",NaN,GC_Mohlke,1,NaN
3,GC_Mohlke:ALT_NC000001.11|230158967|C|A|Mohlke...,GTTTACCCAGCCGTGGGAAAGGACGCTGTACCCCTGCCCTATTGGC...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,230159136,230159406,+,[indel],[192],[NC_000001.11:230159328:TCTTAAAGTGTTCAGCACTCCC:T],[alt],NaN,GC_Mohlke,1,NaN
4,GC_Mohlke:REF_NC000001.11|230161389|C|T|Mohlke...,CCTCAACTCTCCACATGCCCCAGTAGCATAGACCAGCTTCCTTACA...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,230161269,230161539,+,[SNV],[120],[NC_000001.11:230161389:C:T],[ref],NaN,GC_Mohlke,1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80210,GC_Glut_Chengyu:Glut|chr7:31260964-31261233|+|...,TTTCTTATACTGTATGTTTAAAGATATAGACAATGCATATACAAGT...,element,element inactive control,NaN,GRCh38,chr7,31260963,31261233,NaN,None,None,None,None,NaN,GC_Glut_Chengyu,7,"[GC_Glut_Chengyu, GC_GABA_Chengyu]"
80211,C_negative_heart_MK:tile_8033_chr11_69157415_6...,TTTGCAGATTACGTCAGCGCTGGCGAGCAGGGGGCGCGGGGACAGC...,element,element inactive control,general controls IGVF year 1 design 2023,GRCh38,chr11,69157414,69157684,.,None,None,None,None,NaN,C_negative_heart_MK,5,"[C_negative_heart_MK, C_negative_neuron_MK]"
80212,C_negative_heart_MK:tile_22216_chr2_85772228_8...,TTTGTGCCGCCGGCGAGGACTGACACCAGGTTAAGTCGCCTGGCGG...,element,element inactive control,general controls IGVF year 1 design 2023,GRCh38,chr2,85772227,85772497,.,None,None,None,None,NaN,C_negative_heart_MK,5,"[C_negative_heart_MK, C_negative_neuron_MK]"
80213,C_positive_neuron_MK:tile_37243_chr6_97306567_...,TTTTTATTGTACTAATCAGTCTCAAGATGTTCTCTGTGTTAATTGG...,variant,variant positive control,Michael Kosicki,GRCh38,chr6,97306566,97306836,+,[SNV],[244],[NC_000006.12:97306810:G:C],[alt],NaN,C_positive_neuron_MK,1,"[C_positive_neuron_MK, MK]"


In [ ]:
import json

def write_data_with_json(data_df, file_path, interesting_columns=['name', 'sequence', 'category', 'class',
       'source', 'ref', 'chr', 'start', 'end', 'strand', 'variant_class', 'variant_pos', 'SPDI', 'allele',
       'info']):
    """
    Write data with json dumps (for lists)
    """
    # Convert lists/arrays to JSON strings
    for column in data_df[interesting_columns].columns:
        if data_df[column].dtype == 'object' and column in ['allele', 'SPDI', 'variant_class']:
            data_df[column] = data_df[column].apply(lambda x: json.dumps(x) if isinstance(x, (list, np.ndarray)) else x)
    data_df[interesting_columns].to_csv(file_path, sep='\t', index=False, na_rep='NA')
    return data_df

In [ ]:
write_data_with_json(concatenated_metadata_df_deduplicated, 'MPRA_80215.metadata.test.tsv', interesting_columns=interesting_columns)

,name,sequence,category,class,source,ref,chr,start,end,strand,variant_class,variant_pos,SPDI,allele,info,tmp_label,na_count,tmp_label_new
0,GC_Mohlke:REF_NC000001.11|159752292|A|G|Mohlke...,ATACATCCTTTAATTTGTTCCTACATCTTGCTTGGATTTTCCCCTG...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,159752058,159752328,+,"[""SNV""]",[234],"[""NC_000001.11:159752292:A:G""]","[""ref""]",NaN,GC_Mohlke,1,NaN
1,GC_Mohlke:REF_NC000001.11|230158967|C|A|Mohlke...,TGTGTCTGGTGAGGTTGCTGACACTGCTTTTGGATGAGAGAGAGAG...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,230159023,230159293,+,"[""SNV""]",[145],"[""NC_000001.11:230159168:C:T""]","[""ref""]",NaN,GC_Mohlke,1,NaN
2,GC_Mohlke:REF_NC000001.11|230158967|C|A|Mohlke...,GTTTACCCAGCCGTGGGAAAGGACGCTGTACCCCTGCCCTATTGGC...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,230159136,230159406,+,"[""SNV"", ""indel""]","[32, 192]","[""NC_000001.11:230159168:C:T"", ""NC_000001.11:2...","[""ref"", ""ref""]",NaN,GC_Mohlke,1,NaN
3,GC_Mohlke:ALT_NC000001.11|230158967|C|A|Mohlke...,GTTTACCCAGCCGTGGGAAAGGACGCTGTACCCCTGCCCTATTGGC...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,230159136,230159406,+,"[""indel""]",[192],"[""NC_000001.11:230159328:TCTTAAAGTGTTCAGCACTCC...","[""alt""]",NaN,GC_Mohlke,1,NaN
4,GC_Mohlke:REF_NC000001.11|230161389|C|T|Mohlke...,CCTCAACTCTCCACATGCCCCAGTAGCATAGACCAGCTTCCTTACA...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,230161269,230161539,+,"[""SNV""]",[120],"[""NC_000001.11:230161389:C:T""]","[""ref""]",NaN,GC_Mohlke,1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80210,GC_Glut_Chengyu:Glut|chr7:31260964-31261233|+|...,TTTCTTATACTGTATGTTTAAAGATATAGACAATGCATATACAAGT...,element,element inactive control,NaN,GRCh38,chr7,31260963,31261233,NaN,None,None,None,None,NaN,GC_Glut_Chengyu,7,"[GC_Glut_Chengyu, GC_GABA_Chengyu]"
80211,C_negative_heart_MK:tile_8033_chr11_69157415_6...,TTTGCAGATTACGTCAGCGCTGGCGAGCAGGGGGCGCGGGGACAGC...,element,element inactive control,general controls IGVF year 1 design 2023,GRCh38,chr11,69157414,69157684,.,None,None,None,None,NaN,C_negative_heart_MK,5,"[C_negative_heart_MK, C_negative_neuron_MK]"
80212,C_negative_heart_MK:tile_22216_chr2_85772228_8...,TTTGTGCCGCCGGCGAGGACTGACACCAGGTTAAGTCGCCTGGCGG...,element,element inactive control,general controls IGVF year 1 design 2023,GRCh38,chr2,85772227,85772497,.,None,None,None,None,NaN,C_negative_heart_MK,5,"[C_negative_heart_MK, C_negative_neuron_MK]"
80213,C_positive_neuron_MK:tile_37243_chr6_97306567_...,TTTTTATTGTACTAATCAGTCTCAAGATGTTCTCTGTGTTAATTGG...,variant,variant positive control,Michael Kosicki,GRCh38,chr6,97306566,97306836,+,"[""SNV""]",[244],"[""NC_000006.12:97306810:G:C""]","[""alt""]",NaN,C_positive_neuron_MK,1,"[C_positive_neuron_MK, MK]"
